# VVRAG SlideVQA ingestion (free GPU) — decks -> OCR -> ColQwen embeddings
Self-contained: run top to bottom. Clones the repo, installs deps, materializes a
**SlideVQA** subset (8 presentation decks / 40 QA items) as multi-page PDFs in
`data/raw/` + `data/gold.json`, OCRs the slide images with Tesseract, builds real
ColQwen embeddings into `data/index`, verifies the manifest against the gold set,
and zips `data/index` + `data/gold.json` for download.

**Runtime**: ~30-60 min on a free T4 (OCR + ColQwen over ~100-160 slide pages).
Re-running is safe (ingest is resumable; finished pages are skipped).

After downloading `slidevqa_index.zip` to your laptop, unzip it into the repo
root (giving `data/index/` + `data/gold.json`) and run Test Run 6 with:
```
python scripts/run_test5.py --index data/index
```

In [ ]:
%cd /content
!if [ -d altVRAG ]; then git -C altVRAG pull -q; else git clone https://github.com/BiprashPandey/altVVRAG.git altVRAG; fi
%cd /content/altVRAG
!pip install -q -e . --no-deps
!ls -lh data/raw

## 0. GPU, deps, OCR
Runtime > Change runtime type > T4 GPU is required. `datasets` + `pyarrow` are
needed only by this notebook (SlideVQA streaming); everything else matches the
sample ingest notebook. Tesseract is mandatory: SlideVQA deck pages are pure
images with **no text layer**, so all word boxes come from OCR.

In [ ]:
!nvidia-smi || echo 'NO GPU VISIBLE -> Runtime > Change runtime type > T4 GPU'
%pip install -q torch --index-url https://download.pytorch.org/whl/cu121
%pip install -q colpali-engine qdrant-client pymupdf pillow pytesseract transformers groq openai google-genai
%pip install -q datasets pyarrow  # SlideVQA streaming only
!pip install -q -U torchao  # peft requires torchao for the ColQwen LoRA adapter
!apt-get install -y tesseract-ocr > /dev/null 2>&1
!tesseract --version | head -1  # slide images have no text layer: OCR must work
!python -c "import torch; print('torch', torch.__version__, '| cuda build:', torch.version.cuda, '| available:', torch.cuda.is_available())"
!python -c "from vvrag.ingest.embed import probe_environment; probe_environment()"

## 1. Verify the retriever checkpoint
`vidore/colqwen2-v1.0` is public (metadata lookup, downloads nothing). If it
fails, pick a working ColPali checkpoint and pass `--model-id` in the ingest cell.

In [ ]:
from huggingface_hub import HfApi
info = HfApi().model_info("vidore/colqwen2-v1.0")  # metadata only, no download
print("id:", info.id, "| private:", info.private, "| gated:", getattr(info, "gated", None))

## 2. Subset size (edit here)
`N_DECKS` decks x ~15 pages each and `N_QA` graded QA items. The dataset is
streamed; `--max-rows` caps how many rows (one row = one QA item + the full deck
images) are scanned before giving up on unseen decks. 40 QA items ~= 2 free-tier
Gemini quota-days for the live Test Run 6 eval afterwards.

In [ ]:
N_DECKS = 8    # number of distinct presentation decks to materialize
N_QA = 40      # max QA items kept in gold.json
MAX_ROWS = 150  # streaming scan cap (rows = QA items, each carries its deck's images)

## 3. Materialize SlideVQA -> `data/raw/<deck>.pdf` + `data/gold.json`
Uses the ungated `Ahren09/SlideVQA` mirror (the script tries the official gated
`NTT-hil-insight/SlideVQA` first and falls through automatically; no HF token
needed). Each deck becomes ONE multi-page PDF of its slide images (1024x576);
`gold.json` records question, all answer variants, gold page (0-indexed) and
`doc_id` per QA item.

In [ ]:
!python scripts/prepare_slidevqa.py --out data --n-decks {N_DECKS} --n-qa {N_QA} --max-rows {MAX_ROWS}
!ls -lh data/raw
!python -c "import json; g=json.load(open('data/gold.json')); print(len(g), 'QA items'); print('sample:', json.dumps(g[0], indent=1)[:400])"

## 4. Archive the sample docs (SlideVQA-only index)
The repo ships `data/raw/sample.pdf` + `sample.png`; move them aside so the
exported index contains ONLY the SlideVQA decks (keeps the local sample index
reproducible and retrieval clean).

In [ ]:
!mkdir -p data/raw_samples_archive
!mv -f data/raw/sample.pdf data/raw/sample.png data/raw_samples_archive/ 2>/dev/null; echo 'remaining in data/raw:'
!ls -lh data/raw

## 5. Ingest: OCR + ColQwen embeddings + Qdrant + SQLite
Do NOT pass `--skip-ocr`: deck pages are images; Tesseract supplies the word
boxes that sentence segmentation and quote verification depend on. Resumable —
if the run dies, re-run this cell and finished pages are skipped. Expect a few
seconds per page on T4 (~100-160 pages total).

In [ ]:
!python -m vvrag.ingest.run_ingest --input data/raw --out data/index --embedder colqwen

## 6. Verify the index against the gold set
Checks: manifest page count matches ingested decks, every page has OCR words,
one `.npy` embedding per page, and every `gold.json` `doc_id` exists in
`meta.db` with the gold page present.

In [ ]:
!cat data/index/manifest.json
!ls data/index/embeddings/ | head
import json, sqlite3
from pathlib import Path
m = json.load(open('data/index/manifest.json'))
gold = json.load(open('data/gold.json'))
con = sqlite3.connect('data/index/meta.db')
docs = [d for d in con.execute('SELECT DISTINCT doc_id FROM pages')]
docs = sorted(d[0] for d in docs)
print('docs:', docs)
assert len(docs) >= N_DECKS, (len(docs), N_DECKS)
# per-page OCR word counts (slides need > 20 words or OCR failed)
bad = []
for doc, page, n in con.execute(
        'SELECT doc_id, page, COUNT(*) FROM word_boxes GROUP BY doc_id, page'):
    if n < 20:
        bad.append((doc, page, n))
print('pages with <20 OCR words:', bad or 'none')
# embeddings: one npy per page
n_emb = len(list(Path('data/index/embeddings').glob('*.npy')))
n_pages = m['stats']['pages']
print(f'pages={n_pages} embeddings={n_emb}')
assert n_emb >= n_pages, (n_emb, n_pages)
# every gold doc_id + gold page must exist
missing = []
for g in gold:
    row = con.execute('SELECT 1 FROM pages WHERE doc_id=? AND page=?',
                      (g['doc_id'], g['page'])).fetchone()
    if row is None:
        missing.append((g['qid'], g['doc_id'], g['page']))
print('gold items with missing page:', missing or 'none')
assert not missing, missing[:5]
print(f'OK: {len(docs)} decks, {n_pages} pages, {len(gold)} gold QA items verified')

## 7. Export: download `slidevqa_index.zip`
Download via the Files panel (left pane), then on your laptop:
1. unzip into the altVRAG repo root -> `data/index/` + `data/gold.json`
2. Test Run 6 (live Gemini, paced; quota is 20 req/day/model — use
   `--resume <dir> <model>` to continue across days):
```
python scripts/run_test5.py --index data/index
python scripts/make_test5_pdf.py <rows_dir> testRun6.pdf
```

In [ ]:
!zip -qr slidevqa_index.zip data/index data/gold.json && ls -lh slidevqa_index.zip